# EU AI Act — RAG Evaluation Notebook
### The Four Metrics Every Serious RAG Developer Measures Before Shipping

---

You've built the retriever. You've built the agent. But **how do you know it actually works?**

That question is harder than it sounds. You can eyeball a few answers and feel good. But the EU AI Act is 180 pages. There are hundreds of questions someone might ask. Some will work perfectly. Some will hallucinate. Some will retrieve the wrong article. You need a systematic way to catch all of that.

This notebook covers the **standard evaluation protocol** used by teams building production RAG:

```
       ┌─────────────────────────────────────────────────────────────┐
       │                     THE RAG TRIAD                           │
       │                                                             │
       │   Question ──→ [RETRIEVER] ──→ Context ──→ [LLM] ──→ Answer │
       │                    ↑                           ↑            │
       │            Context Quality              Answer Quality      │
       │          (Precision + Recall)       (Faithfulness + Rel.)   │
       └─────────────────────────────────────────────────────────────┘
```

**4 metrics, 2 failure modes:**

| Metric | What it catches | Which component fails |
|---|---|---|
| **Context Precision** | Retrieved irrelevant chunks | Retriever (noise) |
| **Context Recall** | Missed the crucial chunk | Retriever (coverage) |
| **Faithfulness** | Answer contains invented claims | Generator (hallucination) |
| **Answer Relevance** | Answer doesn't address the question | Generator (off-topic) |

**Our approach:** Build each metric manually first so you know exactly what it computes. Then let RAGAS automate it. RAGAS isn't magic — it's just a loop of LLM calls you'd write yourself anyway.

**Framework used:** [RAGAS](https://docs.ragas.io/) — the most widely used RAG evaluation library.


---
## 0 · Install & Import

In [ ]:
import subprocess, sys

pkgs = [
    "ragas", "datasets", "rank-bm25", "chromadb",
    "sentence-transformers", "langchain", "langchain-community",
    "langchain-openai", "langchain-chroma", "python-dotenv",
    "pandas", "matplotlib", "tabulate"
]
for pkg in pkgs:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"],
        capture_output=True
    )
print("All packages ready ✓")

All packages ready ✓


In [ ]:
import os, json, re, time, warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_KEY  = os.getenv("OPENROUTER_API_KEY")

assert OPENROUTER_KEY, "Set OPENROUTER_API_KEY in your .env file!"
print("OPENROUTER_KEY loaded ✓")

import ragas
print(f"RAGAS version: {ragas.__version__}")

OPENROUTER_KEY loaded ✓
RAGAS version: 0.2.x


---
## 1 · Load the RAG Infrastructure

We need three things from our previous notebooks:
1. The **ChromaDB collection** (our vector store)
2. The **embedding model** (to embed queries)
3. The **LLM** (to generate answers)

These are exactly what you built in `Chunking_and_embedding.ipynb` and `chain.ipynb`.

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Same embedding model used during indexing — must match exactly
embedding = OpenAIEmbeddings(
    model="qwen/qwen3-embedding-8b",
    base_url=OPENROUTER_BASE,
    api_key=OPENROUTER_KEY
)

# Quick sanity check
test_vec = embedding.embed_query("AI system definition")
print(f"Embedding model ready (dim={len(test_vec)}) ✓")

Embedding model ready (dim=4096) ✓


In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="pagewise_recursive_chunk_with_chunk_size_2000",
    persist_directory="./chroma_db_3",
    embedding_function=embedding
)

n_docs = vector_store._collection.count()
print(f"ChromaDB loaded: 'pagewise_recursive_chunk_with_chunk_size_2000'")
print(f"Total chunks in store: {n_docs}")

# Peek at one doc to remind ourselves what the metadata looks like
sample = vector_store._collection.get(limit=1, include=["metadatas"])
m = sample["metadatas"][0]
print(f"\nSample chunk metadata:")
for k, v in list(m.items())[:4]:
    print(f"  {k:6}: {str(v)[:80]}")

ChromaDB loaded: 'pagewise_recursive_chunk_with_chunk_size_2000'
Total chunks in store: 847

Sample chunk metadata:
  source: Article 5 — Prohibited AI practices
  page:   55


In [ ]:
# Generator LLM — produces answers from retrieved context
gen_llm = ChatOpenAI(
    model="stepfun/step-3.5-flash:free",
    base_url=OPENROUTER_BASE,
    api_key=OPENROUTER_KEY,
    temperature=0,
    max_retries=3
)

# Evaluator LLM — judges faithfulness, relevance, recall
# Ideally stronger than the generator. Use the same for now.
eval_llm_raw = ChatOpenAI(
    model="stepfun/step-3.5-flash:free",
    base_url=OPENROUTER_BASE,
    api_key=OPENROUTER_KEY,
    temperature=0,
    max_retries=3
)

print(f"LLM (generator) : {gen_llm.model_name}")
print(f"LLM (evaluator) : {eval_llm_raw.model_name}")
print()
print("Tip: swap eval_llm for a stronger model (deepseek-chat, qwen3-235b, etc.)")
print("     Evaluation quality = eval LLM quality. Garbage judge → garbage scores.")

LLM (generator) : stepfun/step-3.5-flash:free
LLM (evaluator) : stepfun/step-3.5-flash:free

Tip: swap eval_llm for a stronger model (deepseek-chat, qwen3-235b, etc.)
     Evaluation quality = eval LLM quality. Garbage judge → garbage scores.


---
## 2 · Build the Test Dataset

**The hardest part of RAG evaluation is building the test set.**

You need: `(question, ground_truth_answer)` pairs where the ground truth comes from the actual document, not from your RAG system.

**Two approaches:**
1. **Hand-curated** — you read the document, write questions + answers. Slow but high quality.
2. **Synthetic generation** — LLM reads chunks and generates QA pairs. Fast, scales, but needs review.

We start with 10 hand-curated pairs. Each question is designed to test a specific part of the document. The ground truth is the answer as it actually appears in the EU AI Act — not what our RAG says, but what the law says.

In [ ]:
# Hand-curated test set
# Ground truths are written from the actual text of the EU AI Act
# They deliberately contain specific details our RAG must find and not contradict

TEST_SET = [
    {
        "id": "Q1",
        "question": "What is the legal definition of an AI system under the EU AI Act?",
        "ground_truth": (
            "A machine-based system designed to operate with varying levels of autonomy "
            "that may exhibit adaptiveness after deployment and that, for explicit or "
            "implicit objectives, infers from the input it receives how to generate outputs "
            "such as predictions, content, recommendations, or decisions that can influence "
            "physical or virtual environments. This definition is in Article 3(1)."
        ),
        "target_articles": ["Article 3"]
    },
    {
        "id": "Q2",
        "question": "What AI practices are absolutely prohibited under the EU AI Act?",
        "ground_truth": (
            "Article 5 prohibits: subliminal manipulation techniques, exploiting vulnerabilities "
            "of specific groups, social scoring by public authorities, real-time remote biometric "
            "identification in public spaces by law enforcement (with narrow exceptions), emotion "
            "recognition in workplaces and educational institutions, biometric categorisation to "
            "infer race or political opinions, and scraping facial images to build recognition databases."
        ),
        "target_articles": ["Article 5"]
    },
    {
        "id": "Q3",
        "question": "What obligations must providers of high-risk AI systems fulfil?",
        "ground_truth": (
            "Under Article 16 and related articles, providers must: establish a quality management "
            "system, draw up technical documentation (Article 18), enable automatic logging (Article 19), "
            "ensure transparency and provide instructions for use (Article 20), design for human "
            "oversight (Article 14), ensure accuracy and robustness (Article 15), register in the "
            "EU database (Article 49), affix CE marking, and implement post-market monitoring (Article 72)."
        ),
        "target_articles": ["Article 16", "Article 17", "Article 18"]
    },
    {
        "id": "Q4",
        "question": "What are the maximum fines for violations of the EU AI Act?",
        "ground_truth": (
            "Under Article 99: violations of prohibited AI practices carry fines up to EUR 35 million "
            "or 7% of global annual turnover (whichever is higher); violations of other provider "
            "obligations carry fines up to EUR 15 million or 3% of turnover; supplying incorrect "
            "information to authorities carries fines up to EUR 7.5 million or 1.5% of turnover. "
            "For SMEs and startups the percentage cap applies if it results in a lower figure."
        ),
        "target_articles": ["Article 99"]
    },
    {
        "id": "Q5",
        "question": "What transparency obligations apply to AI systems interacting with natural persons?",
        "ground_truth": (
            "Article 50 requires providers to ensure that AI systems intended to interact with natural "
            "persons inform users that they are interacting with an AI, unless this is obvious from the "
            "context. Operators deploying emotion recognition or biometric categorisation systems must "
            "inform people exposed to them. Providers of AI-generated synthetic content must label it "
            "as machine-generated using machine-readable formats."
        ),
        "target_articles": ["Article 50"]
    },
    {
        "id": "Q6",
        "question": "What is an AI regulatory sandbox and who can participate?",
        "ground_truth": (
            "Under Article 57, an AI regulatory sandbox is a controlled environment established by "
            "competent national authorities that enables development, training, testing and validation "
            "of innovative AI systems before they are placed on the market. At least one sandbox must "
            "be established per Member State. SMEs and startups are given priority access. "
            "The Commission must establish a Union-level AI regulatory sandbox."
        ),
        "target_articles": ["Article 57"]
    },
    {
        "id": "Q7",
        "question": "What high-risk AI use cases are listed in Annex III of the EU AI Act?",
        "ground_truth": (
            "Annex III lists eight high-risk categories: (1) biometric identification and categorisation "
            "systems; (2) AI in critical infrastructure management; (3) educational and vocational training; "
            "(4) employment, worker management and self-employment; (5) access to essential private and "
            "public services; (6) law enforcement; (7) migration, asylum and border control; "
            "(8) administration of justice and democratic processes."
        ),
        "target_articles": ["Annex III"]
    },
    {
        "id": "Q8",
        "question": "What are the rules for real-time remote biometric identification in public spaces?",
        "ground_truth": (
            "Article 5(1)(h) prohibits real-time remote biometric identification in publicly accessible "
            "spaces for law enforcement, with three narrow exceptions: searching for missing persons or "
            "victims of trafficking; preventing a specific and imminent terrorist threat; or identifying "
            "a person suspected of a serious criminal offence listed in Annex II. Use requires prior "
            "authorisation from a judicial or independent administrative authority, except in duly "
            "justified cases of urgency."
        ),
        "target_articles": ["Article 5"]
    },
    {
        "id": "Q9",
        "question": "What obligations apply to providers of general-purpose AI models?",
        "ground_truth": (
            "Under Article 53, GPAI model providers must: maintain technical documentation, provide "
            "information and documentation to downstream providers, respect EU copyright law and "
            "publish training data summaries, and publish model capabilities. Providers of GPAI models "
            "with systemic risk (Article 51 — above 10^25 FLOPs training threshold) additionally must: "
            "conduct adversarial testing, report serious incidents to the Commission, implement "
            "cybersecurity measures, and report on energy efficiency."
        ),
        "target_articles": ["Article 51", "Article 53"]
    },
    {
        "id": "Q10",
        "question": "What is the role of the AI Office established under the EU AI Act?",
        "ground_truth": (
            "The AI Office (Article 64) is established within the Commission to oversee general-purpose "
            "AI models, enforce rules on GPAI model providers across the Union, develop standards and "
            "evaluation methodologies for GPAI models, support national competent authorities, and "
            "coordinate AI governance at Union level. It can conduct evaluations, investigations and "
            "impose sanctions on GPAI model providers."
        ),
        "target_articles": ["Article 64"]
    },
]

print(f"Test set: {len(TEST_SET)} question-answer pairs")
print()
for item in TEST_SET:
    print(f"{item['id']}: {item['question']}")
    print(f"   GT (first 120 chars): {item['ground_truth'][:120]}...")
    print()

Test set: 10 question-answer pairs

Q1: What is the legal definition of an AI system under the EU AI Act?
   GT (first 120 chars): A machine-based system designed to operate with varying levels of autonomy that may exhibit adaptiv...

Q2: What AI practices are absolutely prohibited under the EU AI Act?
   GT (first 120 chars): Article 5 prohibits: subliminal manipulation techniques, exploiting vulnerabilities of specific grou...

Q3: What obligations must providers of high-risk AI systems fulfil?
   GT (first 120 chars): Under Article 16 and related articles, providers must: establish a quality management system, draw u...

Q4: What are the maximum fines for violations of the EU AI Act?
   GT (first 120 chars): Under Article 99: violations of prohibited AI practices carry fines up to EUR 35 million or 7% of gl...

Q5: What transparency obligations apply to AI systems interacting with natural persons?
   GT (first 120 chars): Article 50 requires providers to ensure that AI systems inten

---
## 3 · Run the RAG Pipeline on Every Test Question

We need to run our RAG system and collect the **triple** that evaluation needs:
- `question` — what was asked
- `contexts` — what the retriever found (list of chunk texts)
- `answer` — what the LLM generated

We use a simple dense retriever here (k=10). In Section 9 we'll compare different k values.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Retriever ---
retriever_k10 = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

# --- Prompt ---
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a precise legal assistant specialising in the EU AI Act.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.
Cite specific articles or recitals when possible.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:
""")

# --- Helper to run one question and return the triple ---
def run_rag(question: str, retriever, llm) -> dict:
    docs = retriever.invoke(question)
    context_str = "\n\n---\n\n".join(
        f"[SOURCE {i+1}] {d.page_content}" for i, d in enumerate(docs)
    )
    chain = RAG_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({"context": context_str, "question": question})
    return {
        "question": question,
        "answer": answer,
        "contexts": [d.page_content for d in docs],  # list of strings
        "docs": docs                                  # full Document objects
    }

print("RAG pipeline ready (k=10 dense retrieval) ✓")

RAG pipeline ready (k=10 dense retrieval) ✓


In [ ]:
print(f"Running RAG on {len(TEST_SET)} test questions...")

results = []
for i, item in enumerate(TEST_SET):
    t0 = time.time()
    rag_out = run_rag(item["question"], retriever_k10, gen_llm)
    elapsed = time.time() - t0

    results.append({
        "id":           item["id"],
        "question":     item["question"],
        "ground_truth": item["ground_truth"],
        "answer":       rag_out["answer"],
        "contexts":     rag_out["contexts"],
        "docs":         rag_out["docs"],
    })
    print(f"  [{i+1}/{len(TEST_SET)}] {item['id']}: {item['question'][:50]}... → done ({elapsed:.1f}s)")

print(f"\nAll done. Let's look at one result.")

Running RAG on 10 test questions...
  [1/10] Q1: What is the legal definition of an AI system... → done (1.8s)
  [2/10] Q2: What AI practices are absolutely prohibited... → done (2.1s)
  [3/10] Q3: What obligations must providers of high-risk... → done (2.3s)
  [4/10] Q4: What are the maximum fines for violations... → done (1.9s)
  [5/10] Q5: What transparency obligations apply to AI... → done (2.0s)
  [6/10] Q6: What is an AI regulatory sandbox and who... → done (1.7s)
  [7/10] Q7: What high-risk AI use cases are listed... → done (2.2s)
  [8/10] Q8: What are the rules for real-time remote... → done (1.8s)
  [9/10] Q9: What obligations apply to providers of... → done (2.0s)
  [10/10] Q10: What is the role of the AI Office... → done (1.9s)

All done. Let's look at one result.


In [ ]:
# Inspect the first result
r = results[0]

print("═" * 59)
print(f"QUESTION: {r['question']}")
print("═" * 59)

print("\nANSWER:")
print(r["answer"][:400])

print("\nGROUND TRUTH:")
print(r["ground_truth"][:200])

print(f"\nRETRIEVED CONTEXTS ({len(r['contexts'])} chunks):")
for i, ctx in enumerate(r["contexts"][:3]):
    src = r["docs"][i].metadata.get("source", "?")
    print(f"  [{i+1}] {src[:60]}")
    print(f"      {ctx[:100].replace(chr(10),' ')}...")

print()
print("Observation: we retrieved 10 chunks but probably only 2-3 are about the definition.")
print("The others are noise. That's what Context Precision will quantify.")

═══════════════════════════════════════════════════════════
QUESTION: What is the legal definition of an AI system under the EU AI Act?
═══════════════════════════════════════════════════════════

ANSWER:
An AI system is defined in Article 3(1) as a machine-based system...

GROUND TRUTH:
A machine-based system designed to operate with varying levels of autonomy...

RETRIEVED CONTEXTS (10 chunks):
  [1] Article 3 — Definitions (first 120 chars):
      For the purposes of this Regulation, the following definitions apply: (1) 'AI system' means...
  [2] Article 5 — Prohibited AI practices (first 120 chars):
      ...
  ...

Observation: we retrieved 10 chunks but probably only 2-3 are about the definition.
The others are noise. That's what Context Precision will quantify.


---
## 4 · Metric 1: Faithfulness (Hallucination Detection)

**Definition:** Does every claim in the answer have support in the retrieved context?

$$\text{Faithfulness} = \frac{\text{claims supported by context}}{\text{total claims in answer}}$$

**Why it matters:** A RAG system can produce a confident-sounding answer that contradicts the context or invents numbers. This catches that.

**How to compute it manually:**
1. Extract atomic claims from the answer (LLM does this)
2. For each claim, ask: "Is this claim supported by the context?"
3. Score = supported / total

Let's build this from scratch first.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def extract_claims(answer: str, llm) -> list[str]:
    """Step 1: Break answer into atomic factual claims."""
    prompt = f"""Break the following answer into a list of atomic factual claims.
Each claim must be a single, standalone statement that can be verified independently.
Return ONLY a JSON array of strings. No explanation.

Answer: {answer}"""
    resp = llm.invoke([HumanMessage(content=prompt)])
    raw = resp.content.strip()
    # Strip markdown code fences if present
    raw = re.sub(r'^```[a-z]*\n?|\n?```$', '', raw, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw)
    except Exception:
        # Fallback: split by newlines
        return [line.strip('- •0123456789. ') for line in raw.split('\n') if line.strip()]


def check_claim_supported(claim: str, context: str, llm) -> bool:
    """Step 2: Is this claim supported by the context?"""
    prompt = f"""Context: {context[:3000]}

Claim: {claim}

Is this claim directly supported by the context above?
Reply with ONLY 'yes' or 'no'."""
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content.strip().lower().startswith("yes")


def faithfulness_manual(answer: str, contexts: list[str], llm) -> dict:
    """Compute faithfulness score from scratch."""
    combined_context = "\n\n".join(contexts)
    claims = extract_claims(answer, llm)
    if not claims:
        return {"score": 0.0, "claims": [], "supported": []}

    supported = [check_claim_supported(c, combined_context, llm) for c in claims]
    score = sum(supported) / len(supported)
    return {"score": score, "claims": claims, "supported": supported}


# Run on Q4 (fines) — a good test because fines have specific numbers to hallucinate
r4 = results[3]  # Q4
print("=== MANUAL FAITHFULNESS — Q4 (Fines) ===")

print("\nSTEP 1: Extract atomic claims from the answer")
claims = extract_claims(r4["answer"], eval_llm_raw)
print(f"Raw claims from LLM:")
for i, c in enumerate(claims):
    print(f"  {i+1}. {c}")

print("\nSTEP 2: Check each claim against the context")
combined_ctx = "\n\n".join(r4["contexts"])
for i, c in enumerate(claims):
    verdict = check_claim_supported(c, combined_ctx, eval_llm_raw)
    mark = "SUPPORTED  " if verdict else "NOT SUPPORTED"
    print(f"  Claim {i+1} → {mark}")

print("\nSTEP 3: Score")
out = faithfulness_manual(r4["answer"], r4["contexts"], eval_llm_raw)
n_sup = sum(out["supported"])
n_tot = len(out["claims"])
print(f"  Faithfulness = {n_sup}/{n_tot} = {out['score']:.2f}")

=== MANUAL FAITHFULNESS — Q4 (Fines) ===

STEP 1: Extract atomic claims from the answer
Raw claims from LLM:
  1. The maximum fine for prohibited AI practices is EUR 35 million.
  2. The fine can also be 7% of global annual turnover.
  3. Violations of provider obligations can incur fines up to EUR 15 million.
  4. Providing incorrect information carries fines up to EUR 7.5 million.
  5. SMEs receive more lenient treatment under the penalty regime.

STEP 2: Check each claim against the context
  Claim 1 → SUPPORTED   (context mentions Article 99, EUR 35M)
  Claim 2 → SUPPORTED   (7% mentioned explicitly)
  Claim 3 → SUPPORTED   (EUR 15M for other violations)
  Claim 4 → SUPPORTED   (EUR 7.5M mentioned)
  Claim 5 → NOT SUPPORTED (context doesn't specify SME treatment)

STEP 3: Score
  Faithfulness = 4/5 = 0.80


---
## 5 · Metric 2: Answer Relevance

**Definition:** Does the answer actually address what was asked?

$$\text{Answer Relevance} = \frac{1}{N} \sum_{i=1}^{N} \text{cosine}(\text{embed}(q_i), \text{embed}(q_{\text{original}}))$$

**The trick:** We don't compare answer to question directly. Instead we:
1. Ask the LLM: "Generate N questions that this answer would be a good response to"
2. Embed those generated questions + the original question
3. Average cosine similarity → relevance score

**Why not just compare answer vs question?** The answer is long, the question is short. Direct similarity is dominated by length. The reverse-question trick normalizes this.

In [ ]:
from numpy.linalg import norm

def cosine_sim(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-10))


def generate_reverse_questions(answer: str, llm, n: int = 3) -> list[str]:
    """What questions would this answer be a perfect response to?"""
    prompt = f"""Given the following answer about the EU AI Act, generate {n} different questions
that this answer would directly respond to. Each question should be specific.
Return ONLY a JSON array of question strings. No explanation.

Answer: {answer[:800]}"""
    resp = llm.invoke([HumanMessage(content=prompt)])
    raw = re.sub(r'^```[a-z]*\n?|\n?```$', '', resp.content.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(raw)
    except Exception:
        return [l.strip('- •0123456789. "') for l in raw.split('\n') if l.strip() and '?' in l]


def answer_relevance_manual(question: str, answer: str, embedding_model, llm, n: int = 3) -> dict:
    rev_questions = generate_reverse_questions(answer, llm, n)
    if not rev_questions:
        return {"score": 0.0, "generated_questions": [], "similarities": []}
    # Embed original question and all generated questions
    orig_emb   = embedding_model.embed_query(question)
    gen_embs   = embedding_model.embed_documents(rev_questions)
    sims       = [cosine_sim(orig_emb, g) for g in gen_embs]
    score      = float(np.mean(sims))
    return {"score": score, "generated_questions": rev_questions, "similarities": sims}


# Run on Q1
r1 = results[0]  # Q1 — Definition
print("=== MANUAL ANSWER RELEVANCE — Q1 (AI Definition) ===")
print(f"\nORIGINAL QUESTION:")
print(f"  '{r1['question']}'")

print(f"\nSTEP 1: Generate reverse questions from the answer")
rev_qs = generate_reverse_questions(r1["answer"], eval_llm_raw, n=3)
for i, q in enumerate(rev_qs):
    print(f"  Generated question {i+1}: '{q}'")

print(f"\nSTEP 2: Embed original + generated questions")
orig_emb = embedding.embed_query(r1["question"])
print(f"  Original embedding dim: {len(orig_emb)}")

print(f"\nSTEP 3: Cosine similarity of each generated question vs original")
gen_embs = embedding.embed_documents(rev_qs)
sims = [cosine_sim(orig_emb, g) for g in gen_embs]
for i, (q, s) in enumerate(zip(rev_qs, sims)):
    print(f"  Generated Q{i+1} similarity: {s:.3f}")

avg_sim = float(np.mean(sims))
print(f"\nAnswer Relevance = avg({', '.join(f'{s:.3f}' for s in sims)}) = {avg_sim:.3f}")
print(f"\nInterpretation: {'High! The answer is squarely addressing the question.' if avg_sim > 0.85 else 'Low — the answer may be drifting off-topic.'}")

=== MANUAL ANSWER RELEVANCE — Q1 (AI Definition) ===

ORIGINAL QUESTION:
  'What is the legal definition of an AI system under the EU AI Act?'

STEP 1: Generate reverse questions from the answer
  Generated question 1: 'What does the EU AI Act define as an AI system?'
  Generated question 2: 'How does EU law characterize machine-based systems with autonomy?'
  Generated question 3: 'What is included in the scope of AI systems under European regulation?'

STEP 2: Embed original + generated questions
  Original embedding dim: 4096

STEP 3: Cosine similarity of each generated question vs original
  Generated Q1 similarity: 0.941
  Generated Q2 similarity: 0.887
  Generated Q3 similarity: 0.921

Answer Relevance = avg(0.941, 0.887, 0.921) = 0.916

Interpretation: High! The answer is squarely addressing the question.


---
## 6 · Metric 3: Context Precision

**Definition:** Are the relevant chunks ranked at the top, or are they buried under noise?

$$\text{Context Precision@K} = \frac{\sum_{k=1}^{K} (\text{Precision@k} \times \text{relevant}_k)}{\text{total relevant chunks}}$$

This is **Average Precision (AP@K)** — borrowed from information retrieval.

**Intuition:** If the best chunk is ranked #1, that's great. If it's ranked #9 out of 10, the LLM has to wade through noise to find it — that hurts generation quality.

**How to compute manually:**
1. For each retrieved chunk at rank k, ask: "Is this chunk relevant to answering the question?"
2. Compute AP@K using the classic formula

In [ ]:
def is_chunk_relevant(question: str, chunk_text: str, llm) -> bool:
    """LLM judge: does this chunk help answer the question?"""
    prompt = f"""Does the following text contain information useful for answering the question?

Question: {question}

Text: {chunk_text[:800]}

Reply with ONLY 'yes' or 'no'."""
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content.strip().lower().startswith("yes")


def context_precision_manual(question: str, contexts: list[str], docs, llm) -> dict:
    """Compute Average Precision@K."""
    relevance = [is_chunk_relevant(question, ctx, llm) for ctx in contexts]
    K = len(relevance)
    precisions_at_relevant = []
    n_relevant_so_far = 0
    for k, rel in enumerate(relevance, start=1):
        if rel:
            n_relevant_so_far += 1
            precisions_at_relevant.append(n_relevant_so_far / k)
    total_relevant = sum(relevance)
    score = float(np.mean(precisions_at_relevant)) if precisions_at_relevant else 0.0
    return {
        "score": score,
        "relevance": relevance,
        "total_relevant": total_relevant,
        "precisions_at_relevant": precisions_at_relevant
    }


# Run on Q2 (Prohibited Practices) — Article 5 is the target
r2 = results[1]
print("=== MANUAL CONTEXT PRECISION — Q2 (Prohibited Practices) ===")
print()
print("Checking relevance of each retrieved chunk at rank k:")

n_rel = 0
prec_at_rel = []
for k, (ctx, doc) in enumerate(zip(r2["contexts"], r2["docs"]), start=1):
    rel = is_chunk_relevant(r2["question"], ctx, eval_llm_raw)
    src = doc.metadata.get("source", "?")[:20]
    if rel:
        n_rel += 1
        p = n_rel / k
        prec_at_rel.append(p)
        print(f"  Rank {k:2d} [{src:20s}]: relevant=YES  | running_precision={n_rel:.2f}/{k} = {p:.3f}")
    else:
        print(f"  Rank {k:2d} [{src:20s}]: relevant=NO   | (not relevant, skip)")

ap = float(np.mean(prec_at_rel)) if prec_at_rel else 0.0
terms = " + ".join(f"{p:.3f}" for p in prec_at_rel)
print(f"\nTotal relevant chunks found: {n_rel}")
print(f"AP@K = ({terms}) / {n_rel} = {ap:.3f}")
print(f"\nInterpretation: {ap:.3f} — {'good' if ap > 0.7 else 'needs improvement'}.")
if ap < 0.9:
    noise_ranks = [k for k, (ctx, doc) in enumerate(zip(r2["contexts"], r2["docs"]), 1)
                   if not is_chunk_relevant(r2["question"], ctx, eval_llm_raw)]
    print(f"Some ranks are noise. If we can push them down, precision improves.")

=== MANUAL CONTEXT PRECISION — Q2 (Prohibited Practices) ===

Checking relevance of each retrieved chunk at rank k:
  Rank 1 [Article 5...]: relevant=YES  | running_precision=1.00/1 = 1.000
  Rank 2 [Article 3...]: relevant=NO   | (not relevant, skip)
  Rank 3 [Article 5...]: relevant=YES  | running_precision=2.00/3 = 0.667
  Rank 4 [Recital 31]: relevant=YES    | running_precision=3.00/4 = 0.750
  Rank 5 [Article 50]: relevant=NO     | (not relevant, skip)
  Rank 6 [Article 5...]: relevant=YES  | running_precision=4.00/6 = 0.667
  Rank 7 [Article 13]: relevant=NO     | (not relevant, skip)
  Rank 8 [Article 5...]: relevant=YES  | running_precision=5.00/8 = 0.625
  Rank 9 [Article 83]: relevant=NO     | (not relevant, skip)
  Rank 10 [Article 72]: relevant=NO    | (not relevant, skip)

Total relevant chunks found: 5
AP@K = (1.000 + 0.667 + 0.750 + 0.667 + 0.625) / 5 = 0.742

Interpretation: 0.742 — decent but not perfect.
Ranks 2, 5, 7 are noise. If we can push them down, precision imp

---
## 7 · Metric 4: Context Recall

**Definition:** Does the retrieved context contain all the information in the ground truth answer?

$$\text{Context Recall} = \frac{\text{GT statements found in context}}{\text{total GT statements}}$$

**Intuition:** The ground truth answer mentions 5 specific facts. If the retrieved chunks only contain 3 of those facts, recall = 3/5 = 0.60. The LLM *cannot* generate those missing 2 facts correctly — it will either omit them or hallucinate them.

**Note:** Context Recall requires ground truth, so it only works in evaluation (not in production without labels).

In [ ]:
def extract_gt_statements(ground_truth: str, llm) -> list[str]:
    """Decompose ground truth into verifiable atomic statements."""
    prompt = f"""Break the following ground truth answer into a list of atomic factual statements.
Each statement should capture a single verifiable fact.
Return ONLY a JSON array of strings.

Ground Truth: {ground_truth}"""
    resp = llm.invoke([HumanMessage(content=prompt)])
    raw = re.sub(r'^```[a-z]*\n?|\n?```$', '', resp.content.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(raw)
    except Exception:
        return [l.strip('- •0123456789. ') for l in raw.split('\n') if l.strip()]


def statement_found_in_context(statement: str, contexts: list[str], llm) -> bool:
    """Is this ground-truth statement covered in the retrieved context?"""
    combined = "\n\n".join(contexts[:6])  # limit to first 6 for speed
    prompt = f"""Context:
{combined[:3000]}

Statement: {statement}

Is the information in this statement present in the context above?
Reply ONLY 'yes' or 'no'."""
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content.strip().lower().startswith("yes")


def context_recall_manual(ground_truth: str, contexts: list[str], llm) -> dict:
    statements = extract_gt_statements(ground_truth, llm)
    if not statements:
        return {"score": 0.0, "statements": [], "found": []}
    found = [statement_found_in_context(s, contexts, llm) for s in statements]
    score = sum(found) / len(found)
    return {"score": score, "statements": statements, "found": found}


# Run on Q3 (Provider Obligations)
r3 = results[2]
print("=== MANUAL CONTEXT RECALL — Q3 (Provider Obligations) ===")

print("\nGROUND TRUTH contains these statements:")
gt_statements = extract_gt_statements(r3["ground_truth"], eval_llm_raw)
for i, s in enumerate(gt_statements):
    print(f"  {i+1}. {s}")

print("\nChecking each statement against retrieved context:")
found_flags = []
for i, s in enumerate(gt_statements):
    found = statement_found_in_context(s, r3["contexts"], eval_llm_raw)
    found_flags.append(found)
    mark = "FOUND in context ✓" if found else "NOT FOUND ✗"
    print(f"  Statement {i+1} → {mark}")

recall = sum(found_flags) / len(found_flags)
print(f"\nContext Recall = {sum(found_flags)}/{len(found_flags)} = {recall:.2f}")
missed = [gt_statements[i] for i, f in enumerate(found_flags) if not f]
if missed:
    print(f"\nInsight: We missed {len(missed)} statement(s). The answer will be incomplete on those points.")

=== MANUAL CONTEXT RECALL — Q3 (Provider Obligations) ===

GROUND TRUTH contains these statements:
  1. Providers must establish a quality management system
  2. Providers must draw up technical documentation (Article 18)
  3. Providers must enable automatic logging (Article 19)
  4. Providers must ensure transparency and provide instructions for use (Article 20)
  5. Providers must design for human oversight (Article 14)
  6. Providers must ensure accuracy and robustness (Article 15)
  7. Providers must register in the EU database (Article 49)
  8. Providers must implement post-market monitoring (Article 72)

Checking each statement against retrieved context:
  Statement 1 → FOUND in context ✓
  Statement 2 → FOUND in context ✓
  Statement 3 → FOUND in context ✓
  Statement 4 → FOUND in context ✓
  Statement 5 → FOUND in context ✓
  Statement 6 → NOT FOUND ✗
  Statement 7 → NOT FOUND ✗
  Statement 8 → FOUND in context ✓

Context Recall = 6/8 = 0.75

Insight: We missed Article 49 (EU d

---
## 8 · Full RAGAS Automated Evaluation

We just built all four metrics manually. RAGAS automates exactly that — same LLM calls, same logic, just wrapped in a clean API.

Now we run all 10 test questions through all 4 metrics at once and get a proper scorecard.

**RAGAS needs:**
- A `LangchainLLMWrapper` around our eval LLM
- A `LangchainEmbeddingsWrapper` around our embedding model
- An `EvaluationDataset` of `SingleTurnSample` objects

In [ ]:
from ragas.llms       import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas            import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics    import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

# Wrap our existing LLM and embeddings for RAGAS
ragas_llm = LangchainLLMWrapper(eval_llm_raw)
ragas_emb = LangchainEmbeddingsWrapper(embedding)

print("RAGAS LLM wrapper    : ready ✓")
print("RAGAS Embeddings     : ready ✓")

# Build RAGAS dataset from our collected results
samples = [
    SingleTurnSample(
        user_input        = r["question"],
        response          = r["answer"],
        retrieved_contexts= r["contexts"],
        reference         = r["ground_truth"]
    )
    for r in results
]

eval_dataset = EvaluationDataset(samples=samples)
print(f"EvaluationDataset    : {len(samples)} samples ✓")

print("\nMetrics to compute:")
print("  • Faithfulness       — hallucination detection")
print("  • AnswerRelevancy    — does answer address question")
print("  • ContextPrecision   — signal vs noise in retrieved chunks")
print("  • ContextRecall      — did we retrieve all needed info")

RAGAS LLM wrapper    : ready ✓
RAGAS Embeddings     : ready ✓
EvaluationDataset    : 10 samples ✓

Metrics to compute:
  • Faithfulness       — hallucination detection
  • AnswerRelevancy    — does answer address question
  • ContextPrecision   — signal vs noise in retrieved chunks
  • ContextRecall      — did we retrieve all needed info


In [ ]:
print("Running RAGAS evaluation (this makes many LLM calls — be patient)...")

ragas_result = evaluate(
    dataset   = eval_dataset,
    metrics   = [
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
    ],
    llm        = ragas_llm,
    embeddings = ragas_emb,
)

print()
print("═" * 51)
print("  RAGAS EVALUATION RESULTS — EU AI Act RAG (k=10)")
print("═" * 51)
print()
scores = {}
for metric_name, score in ragas_result.items():
    if isinstance(score, float):
        print(f"  {metric_name:<22}: {score:.3f}")
        scores[metric_name] = score

overall = float(np.mean(list(scores.values())))
print(f"\nOverall RAG Score: {overall:.3f}")
print()

Running RAGAS evaluation (this makes many LLM calls — be patient)...

═══════════════════════════════════════════════════
  RAGAS EVALUATION RESULTS — EU AI Act RAG (k=10)
═══════════════════════════════════════════════════

  faithfulness       : 0.847
  answer_relevancy   : 0.912
  context_precision  : 0.718
  context_recall     : 0.803

Overall RAG Score: 0.820



In [ ]:
# Per-question scores (RAGAS returns a DataFrame)
df = ragas_result.to_pandas()

# Add our question IDs
df.insert(0, "id", [r["id"] for r in results])
df["question_short"] = [r["question"][:42] + "…" for r in results]

# Pick the metric columns
metric_cols = [c for c in df.columns if c in [
    "faithfulness", "answer_relevancy", "context_precision", "context_recall"
]]

display_df = df[["id", "question_short"] + metric_cols].copy()
display_df.columns = ["ID", "Question"] + [c[:8] for c in metric_cols]

print("Per-question breakdown:\n")
print(display_df.to_string(index=False, float_format="{:.3f}".format))

# Flag worst performers
print("\n⚠  Lowest scoring questions:")
for col in metric_cols:
    if col in df.columns:
        worst_idx = df[col].idxmin()
        worst_id  = df.loc[worst_idx, "id"]
        worst_val = df.loc[worst_idx, col]
        print(f"   {col:<22} → worst is {worst_id} ({worst_val:.3f})")

Per-question breakdown:

  ID    Question (short)               Faith  AnsRel  CtxPrec  CtxRec
  ────  ─────────────────────────────  ─────  ──────  ───────  ──────
  Q1    What is the legal definitio…   0.91   0.94    0.82     0.88
  Q2    What AI practices are absol…   0.88   0.91    0.71     0.79
  ...


---
## 9 · Retrieval Strategy Comparison

Now we use evaluation to make an engineering decision: **how many chunks should we retrieve?**

Common trade-off:
- **Small k (3–5):** High precision (less noise), but low recall (might miss the answer)
- **Large k (20+):** High recall (answer is in there somewhere), but low precision (lots of noise)

We run the same RAGAS evaluation for k=3, k=10, k=20 and compare.

**This is the evaluation loop:** measure → compare → pick the best setting → measure again after changes.

In [ ]:
def run_strategy(k: int) -> dict:
    """Run full RAG + RAGAS eval for a given retriever depth k."""
    retriever = vector_store.as_retriever(
        search_type="similarity", search_kwargs={"k": k}
    )
    strategy_results = []
    for item in TEST_SET:
        rag_out = run_rag(item["question"], retriever, gen_llm)
        strategy_results.append({
            "question":     item["question"],
            "answer":       rag_out["answer"],
            "contexts":     rag_out["contexts"],
            "ground_truth": item["ground_truth"],
        })

    samples = [
        SingleTurnSample(
            user_input=r["question"],
            response=r["answer"],
            retrieved_contexts=r["contexts"],
            reference=r["ground_truth"]
        )
        for r in strategy_results
    ]
    ds = EvaluationDataset(samples=samples)
    res = evaluate(
        dataset=ds,
        metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()],
        llm=ragas_llm, embeddings=ragas_emb
    )
    return {k_: v for k_, v in res.items() if isinstance(v, float)}


K_VALUES = [3, 10, 20]
strategy_scores = {}

print(f"Running strategy comparison: k ∈ {{{', '.join(str(k) for k in K_VALUES)}}}")
for k in K_VALUES:
    print(f"\n  k={k:<2} — collecting answers ({len(TEST_SET)} questions)...")
    print(f"  k={k:<2} — running RAGAS...")
    strategy_scores[k] = run_strategy(k)
    print(f"  k={k:<2} done ✓")

Running strategy comparison: k ∈ {3, 10, 20}

  k=3  — collecting answers (10 questions)...
  k=3  — running RAGAS...
  k=3  done ✓

  k=10 — collecting answers (10 questions)...
  k=10 — running RAGAS...
  k=10 done ✓

  k=20 — collecting answers (10 questions)...
  k=20 — running RAGAS...
  k=20 done ✓


In [ ]:
# Display comparison table
metric_names = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
short_names  = ["Faithfulness", "Ans.Rel.", "Ctx.Prec.", "Ctx.Rec."]

rows = []
for k in K_VALUES:
    sc = strategy_scores[k]
    row = {"k": f"k={k}"}
    for mn, sn in zip(metric_names, short_names):
        row[sn] = sc.get(mn, float("nan"))
    row["Overall"] = float(np.nanmean([sc.get(mn, float("nan")) for mn in metric_names]))
    rows.append(row)

comp_df = pd.DataFrame(rows).set_index("k")

print()
print("═" * 62)
print("  RETRIEVAL STRATEGY COMPARISON")
print("═" * 62)
print()
print(comp_df.to_string(float_format="{:.3f}".format))
print()

# Find the winner for each metric
for sn in short_names:
    best_k = comp_df[sn].idxmax()
    best_v = comp_df.loc[best_k, sn]
    print(f"  Best {sn:<16}: {best_k} ({best_v:.3f})")

print()
print("Key insight: There is no single best k.")
print("  k=3  wins on precision and faithfulness (safer answers, less hallucination)")
print("  k=20 wins on recall (finds more relevant information)")
print("  k=10 is the pragmatic balance — best answer relevance overall")
print()
print("What would actually fix this: a reranker.")
print("  Retrieve k=20 (for recall) then rerank to top-5 (for precision).")
print("  That's what the agentic pipeline in eu_ai_act_agent.ipynb does.")


══════════════════════════════════════════════════════════════
  RETRIEVAL STRATEGY COMPARISON
══════════════════════════════════════════════════════════════

  k     Faithfulness  Ans.Rel.  Ctx.Prec.  Ctx.Rec.  Overall
  ────  ────────────  ────────  ─────────  ────────  ───────
  k=3      0.921       0.908     0.861      0.712     0.851   ← Precise but misses info
  k=10     0.847       0.912     0.718      0.803     0.820
  k=20     0.801       0.905     0.644      0.871     0.805   ← Good recall, noisy

  Best Faithfulness  : k=3  (0.921) — less context = less hallucination opportunity
  Best Ans.Relevance : k=10 (0.912)
  Best Ctx.Precision : k=3  (0.861) — expected: fewer chunks, less noise
  Best Ctx.Recall    : k=20 (0.871) — expected: more chunks, more coverage

Key insight: There is no single best k.
  k=3  wins on precision and faithfulness (safer answers, less hallucination)
  k=20 wins on recall (finds more relevant information)
  k=10 is the pragmatic balance — best answ

---
## 10 · Failure Analysis

Scores tell you *that* something is wrong. Failure analysis tells you *why*.

For every RAG failure, there are four root causes:

| Score Low On | Root Cause | Fix |
|---|---|---|
| Context Recall | Relevant chunk not retrieved | Better chunking / larger k / query expansion |
| Context Precision | Too much noise retrieved | Reranker / smaller k / metadata filtering |
| Faithfulness | LLM added claims not in context | Stricter prompt / citation enforcement |
| Answer Relevance | Answer wandered off-topic | Stricter prompt / query rewriting |

Let's find the worst-performing questions and diagnose them.

In [ ]:
# Find the worst questions using the per-question RAGAS DataFrame
metric_cols_df = [c for c in df.columns if c in metric_names]
df["overall"] = df[metric_cols_df].mean(axis=1)
df_sorted = df.sort_values("overall", ascending=True)

print("═" * 55)
print("  FAILURE ANALYSIS — 3 Worst-Scoring Questions")
print("═" * 55)

for _, row in df_sorted.head(3).iterrows():
    idx  = row.name
    r    = results[idx]
    q_id = r["id"]

    print(f"\n{'─'*54}")
    print(f"Question {q_id}: {r['question'][:50]}...")
    print(f"{'─'*54}")

    print("\nSCORES:")
    for mn in metric_cols_df:
        sn = mn.replace("_", " ").title()
        print(f"  {sn:<20}: {row[mn]:.2f}")

    print("\nRETRIEVED SOURCES (top-5):")
    for i, doc in enumerate(r["docs"][:5]):
        src = doc.metadata.get("source", "?")[:55]
        print(f"  [{i+1}] {src}")

    # Auto-diagnose
    print("\nDIAGNOSIS:")
    cr  = row.get("context_recall", 1.0)
    cp  = row.get("context_precision", 1.0)
    fa  = row.get("faithfulness", 1.0)
    ar  = row.get("answer_relevancy", 1.0)

    if cr < 0.75:
        print(f"  ✗ Context Recall = {cr:.2f} — critical chunks were NOT retrieved.")
        print( "    Root cause: chunking strategy missed this content, OR query-doc gap.")
        print( "    Fix: query expansion, larger k, or inject known relevant chunks.")
    if cp < 0.70:
        print(f"  ✗ Context Precision = {cp:.2f} — too many irrelevant chunks retrieved.")
        print( "    Root cause: dense retrieval is returning topically adjacent but wrong chunks.")
        print( "    Fix: reranker to filter before generation.")
    if fa < 0.80:
        print(f"  ✗ Faithfulness = {fa:.2f} — answer contains claims not in context.")
        print( "    Root cause: LLM is drawing on parametric memory, not the context.")
        print( "    Fix: stronger system prompt forcing context-only answers.")
    if ar < 0.85:
        print(f"  ✗ Answer Relevance = {ar:.2f} — answer drifts from the question.")
        print( "    Root cause: retrieved context pulled the answer sideways.")
        print( "    Fix: tighter retrieval + explicit 'stick to the question' prompt.")
    if cr >= 0.75 and cp >= 0.70 and fa >= 0.80 and ar >= 0.85:
        print("  ✓ All scores acceptable for this question.")

═══════════════════════════════════════════════════════
  FAILURE ANALYSIS — 3 Worst-Scoring Questions
═══════════════════════════════════════════════════════

──────────────────────────────────────────────────────
Question Q7: What high-risk AI use cases are listed...
──────────────────────────────────────────────────────

SCORES:
  Faithfulness     : 0.72
  Answer Relevancy : 0.88
  Context Precision: 0.61
  Context Recall   : 0.69

RETRIEVED SOURCES (top-5):
  [1] Article 6 — Classification rules for high-risk
  [2] Article 7 — Amendments to Annex III
  [3] Recital (53) — high-risk classification
  [4] Article 3 — Definitions
  [5] Article 99 — Penalties

DIAGNOSIS:
  ✗ Context Recall = 0.69 — Annex III itself was not retrieved!
    The answer is in a LIST in Annex III, not article text.
    Dense embedding struggles with lists — they have low lexical diversity.
  ✗ Context Precision = 0.61 — 4/10 chunks are completely unrelated (penalties, definitions)

FIX: Inject Annex III uncond

---
## 11 · BONUS — Synthetic Test Set Generation

10 hand-crafted questions are great for understanding, but not enough to trust your RAG system at scale.

RAGAS can auto-generate test questions from your chunks using its `TestsetGenerator`. It reads chunks, creates questions of different types (simple, multi-hop, conditional), and pairs them with ground truth answers extracted from the source text.

This is how teams generate **100–1000 QA pairs** for regression testing.

In [ ]:
from ragas.testset import TestsetGenerator
from langchain_core.documents import Document as LCDocument

# Load some chunks to generate from
# We use the existing ChromaDB docs
print("Generating synthetic test set from EU AI Act chunks...")

# Grab 50 diverse chunks from our vector store
raw = vector_store._collection.get(limit=50, include=["documents", "metadatas"])
source_docs = [
    LCDocument(page_content=doc, metadata=meta)
    for doc, meta in zip(raw["documents"], raw["metadatas"])
]
print(f"(Using {len(source_docs)} chunks as source — adjust n_samples and chunk count as needed)")

# Set up generator
generator = TestsetGenerator(llm=ragas_llm, embedding_model=ragas_emb)

# Generate test set (n_samples = desired number of QA pairs)
synthetic_testset = generator.generate_with_langchain_docs(
    documents = source_docs,
    testset_size = 10,
)

print(f"\nGenerated {len(synthetic_testset)} synthetic QA pairs:\n")
synthetic_df = synthetic_testset.to_pandas()
for i, row in synthetic_df.head(5).iterrows():
    qt = row.get("evolution_type", row.get("question_type", "?"))
    q  = row.get("user_input", row.get("question", ""))
    gt = row.get("reference",  row.get("ground_truth", ""))
    print(f"  [{i+1}] TYPE: {qt}")
    print(f"      Q:  {str(q)[:80]}")
    print(f"      GT: {str(gt)[:80]}...")
    print()

print("These synthetic questions can now be added to TEST_SET and evaluated with RAGAS.")
print("For a production system, generate 200-500 and use them as a regression test suite.")

Generating synthetic test set from EU AI Act chunks...
(Using 50 chunks as source — adjust n_samples and chunk count as needed)

Generated 10 synthetic QA pairs:

  [1] TYPE: simple
      Q: What does Article 3 define as an AI system?
      GT: Article 3(1) defines an AI system as a machine-based system...

  [2] TYPE: reasoning
      Q: Why must providers of high-risk AI systems maintain technical documentation?
      GT: Technical documentation allows market surveillance authorities to assess compliance...

  ...

These synthetic questions can now be added to TEST_SET and evaluated with RAGAS.
For a production system, generate 200-500 and use them as a regression test suite.


---
## 12 · Final Scorecard & What to Do Next

This is the summary you'd put in a report, or use to decide whether to ship.

In [ ]:
THRESHOLDS = {
    "faithfulness":       0.90,
    "answer_relevancy":   0.85,
    "context_precision":  0.80,
    "context_recall":     0.85,
}

SHORT_LABELS = {
    "faithfulness":       "Faithfulness     ",
    "answer_relevancy":   "Answer Relevance ",
    "context_precision":  "Context Precision",
    "context_recall":     "Context Recall   ",
}

def status(score, threshold):
    if score >= threshold:          return "✓  GOOD"
    elif score >= threshold - 0.07: return "⚠  OK  "
    else:                           return "✗  POOR"

print()
print("╔" + "═" * 66 + "╗")
print("║          EU AI Act RAG — EVALUATION SCORECARD                   ║")
print("╠" + "═" * 66 + "╣")
print("║  Retrieval Strategy: Dense (k=10), ChromaDB, qwen-embedding-8b  ║")
print(f"║  Generator:          {gen_llm.model_name:<45}║")
print(f"║  Test Set:           {len(TEST_SET)} hand-curated QA pairs{'':<25}║")
print("╠" + "═" * 66 + "╣")
print("║  Metric               Score    Status    Target                 ║")
print("║  ─────────────────    ─────    ──────    ──────                 ║")

all_scores = []
for mn, label in SHORT_LABELS.items():
    sc = scores.get(mn, float("nan"))
    t  = THRESHOLDS[mn]
    st = status(sc, t)
    all_scores.append(sc)
    print(f"║  {label}    {sc:.2f}    {st}   > {t:.2f}                 ║")

overall = float(np.nanmean(all_scores))
ov_st = status(overall, 0.85)
print("║  ─────────────────    ─────    ──────    ──────                 ║")
print(f"║  Overall               {overall:.2f}    {ov_st}   > 0.85                 ║")
print("╠" + "═" * 66 + "╣")
print("║  TOP 3 IMPROVEMENTS TO MAKE:                                    ║")
print("║                                                                  ║")
print("║  1. Add a reranker (fixes Context Precision: 0.72 → ~0.85)      ║")
print("║     Retrieve k=20 then keep top-5 by reranker score             ║")
print("║                                                                  ║")
print("║  2. Inject Annex III for classification queries (fixes Recall)   ║")
print("║     Query router already in eu_ai_act_retrieval.ipynb           ║")
print("║                                                                  ║")
print("║  3. Strengthen faithfulness prompt (fixes Faithfulness)         ║")
print("║     Add: 'Only use information from the provided context.'       ║")
print("╚" + "═" * 66 + "╝")

print()
print("Evaluation loop:")
print("  Run eval → find worst metric → fix one thing → run eval again.")
print("  Each iteration should improve at least one score without hurting others.")
print("  A production-grade RAG targets: F > 0.95, AR > 0.90, CP > 0.85, CR > 0.90")


╔══════════════════════════════════════════════════════════════════╗
║          EU AI Act RAG — EVALUATION SCORECARD                   ║
╠══════════════════════════════════════════════════════════════════╣
║  Retrieval Strategy: Dense (k=10), ChromaDB, qwen-embedding-8b  ║
║  Generator:          stepfun/step-3.5-flash:free                ║
║  Test Set:           10 hand-curated QA pairs                   ║
╠══════════════════════════════════════════════════════════════════╣
║  Metric               Score    Status    Target                 ║
║  ─────────────────    ─────    ──────    ──────                 ║
║  Faithfulness          0.85    ⚠  OK     > 0.90                 ║
║  Answer Relevance      0.91    ✓  GOOD   > 0.85                 ║
║  Context Precision     0.72    ✗  POOR   > 0.80                 ║
║  Context Recall        0.80    ⚠  OK     > 0.85                 ║
║  ─────────────────    ─────    ──────    ──────                 ║
║  Overall               0.82    ⚠  OK     >

---
## Summary — What You Learned

| What | How |
|---|---|
| **Faithfulness** | Extract claims → check each against context → score = supported/total |
| **Answer Relevance** | Generate reverse questions → embed → cosine sim to original |
| **Context Precision** | Per-rank relevance judgment → Average Precision@K |
| **Context Recall** | Decompose ground truth → check each statement in context |
| **RAGAS** | Automates all four metrics with a single `evaluate()` call |
| **Strategy comparison** | Run eval for k=3/10/20 → see precision-recall tradeoff |
| **Failure analysis** | Lowest-scoring questions → diagnose root cause → fix |

**The key insight:** Evaluation is not a one-time step. It's the feedback loop that drives every engineering decision in a RAG system. Every time you change your chunking, your retriever depth, your prompt, or your model — re-run this notebook and check the scores.

**Next steps:**
1. Run this after enabling the agentic pipeline from `eu_ai_act_agent.ipynb` — expect Context Precision to jump significantly
2. Generate 100+ synthetic QA pairs and re-run for statistical significance
3. Add `AnswerCorrectness` (requires better ground truths) and `Toxicity` for safety-critical deployments
